# Improved Anomaly Detection Pipeline

## Overview
This notebook improves on the baseline (AUC: 0.8573) through:

1. **Rich Feature Engineering** — 33 user-level behavioral features including rating distribution moments, entropy, item deviation, SVD latent embeddings
2. **Ensemble of Supervised Models** — RandomForest + ExtraTrees + GradientBoosting with OOF weight optimization
3. **Class Imbalance Handling** — `class_weight='balanced'` + random oversampling of minority class per fold
4. **Stratified K-Fold Cross-Validation** — 5-fold CV to track AUC, Precision, Recall, F1 and detect overfitting
5. **Optimized Ensemble Blending** — Nelder-Mead weight optimization on OOF predictions

**Result: OOF AUC ≈ 0.944 vs Baseline 0.857**

## Section 1 — Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import entropy, skew, kurtosis
from scipy.sparse import csr_matrix
from scipy.optimize import minimize

from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier,
    IsolationForest
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score
)
from sklearn.utils import resample

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_ITEMS = 1000
N_FOLDS = 5
SVD_COMPONENTS = 15

print("All imports successful.")

All imports successful.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
path = "/content/drive/MyDrive/"
data_path = os.path.join(path, "")

os.chdir("/content/drive/My Drive/cs421 principles of machine learning/group project/leong/iter 2")

Mounted at /content/drive


## Section 2 — Load & Explore Data

In [ ]:
# --- Load Training Data ---
data_train = np.load("training_batch_with_labels.npz")
X_raw      = data_train["X"]   # shape: (interactions, 3) — [user, item, rating]
y_raw      = data_train["y"]   # shape: (users, 2)        — [user, label]

ratings = pd.DataFrame(X_raw, columns=["user", "item", "rating"])
labels  = pd.DataFrame(y_raw, columns=["user", "label"]).set_index("user")

print("=== Training Data ===")
print(f"Total interactions : {len(ratings):,}")
print(f"Unique users       : {ratings['user'].nunique()}")
print(f"Unique items       : {ratings['item'].nunique()}")
print(f"Rating range       : {ratings['rating'].min()} – {ratings['rating'].max()}")
print(f"\nLabel distribution:")
print(labels['label'].value_counts().rename({0: 'Normal', 1: 'Anomalous'}))

imbalance_ratio = (labels['label']==0).sum() / (labels['label']==1).sum()
print(f"\nImbalance ratio (normal:anomalous) = {imbalance_ratio:.1f}:1")

=== Training Data ===
Total interactions : 177,346
Unique users       : 1100
Unique items       : 993
Rating range       : 0 – 5

Label distribution:
label
Normal       1000
Anomalous     100
Name: count, dtype: int64

Imbalance ratio (normal:anomalous) = 10.0:1


## Section 3 — Feature Engineering

We engineer **33 user-level features** across 6 groups:

| Group | Features |
|---|---|
| Basic stats | mean, std, count, min, max, median, range |
| Higher moments | skewness, kurtosis |
| Rating distribution | entropy, extreme/high/low proportions |
| Interaction diversity | unique items, diversity ratio |
| Item deviation | mean/std/abs deviation from global item averages |
| Item popularity | mean/std of rated item popularity |
| Sequential behavior | repeated ratings, consecutive diff std |
| Latent (SVD) | 15-dim truncated SVD on user–item matrix |

In [ ]:
def build_features(
    df: pd.DataFrame,
    global_item_means=None,
    svd_model=None,
    n_items: int = N_ITEMS,
    fit_svd: bool = False,
):
    """
    Build rich user-level features from raw [user, item, rating] interaction data.

    Parameters
    ----------
    df              : DataFrame with columns [user, item, rating]
    global_item_means : pre-computed item means from training (pass None to fit)
    svd_model       : pre-fitted TruncatedSVD (pass None to fit)
    n_items         : total number of items in the dataset
    fit_svd         : if True, fit a new SVD; if False, transform using provided model

    Returns
    -------
    features          : user-indexed DataFrame of features
    global_item_means : item mean ratings (for reuse on test set)
    svd_model         : fitted SVD model (for reuse on test set)
    """

    # ── Helper functions ──────────────────────────────────────────────────────
    def safe_skew(x):
        return float(skew(x)) if len(x) >= 3 else 0.0

    def safe_kurt(x):
        return float(kurtosis(x)) if len(x) >= 4 else 0.0

    def rating_entropy(x):
        """Shannon entropy over the 0–5 rating distribution."""
        counts = np.bincount(x.astype(int), minlength=6)
        p = counts / counts.sum()
        return float(entropy(p + 1e-9))

    def extreme_prop(x):
        """Proportion of 0 or 5 ratings."""
        return float(((x == 0) | (x == 5)).mean())

    def high_prop(x):
        """Proportion of ratings ≥ 4."""
        return float((x >= 4).mean())

    def low_prop(x):
        """Proportion of ratings ≤ 1."""
        return float((x <= 1).mean())

    def repeated_rating_prop(x):
        """Proportion of consecutive identical ratings (bot-like behaviour)."""
        if len(x) < 2:
            return 0.0
        arr = x.values
        return float((arr[1:] == arr[:-1]).mean())

    def consec_diff_std(x):
        """Std of consecutive rating differences."""
        if len(x) < 2:
            return 0.0
        return float(np.std(np.diff(x.values)))

    # ── Group 1: Basic statistics ─────────────────────────────────────────────
    basic = df.groupby("user")["rating"].agg(
        mean_rating   = "mean",
        std_rating    = "std",
        num_ratings   = "count",
        min_rating    = "min",
        max_rating    = "max",
        median_rating = "median",
    ).fillna(0)
    basic["rating_range"] = basic["max_rating"] - basic["min_rating"]

    # ── Group 2: Higher moments ───────────────────────────────────────────────
    moments = df.groupby("user")["rating"].agg(
        skewness = safe_skew,
        kurt     = safe_kurt,
    )

    # ── Group 3: Rating distribution ──────────────────────────────────────────
    dist = df.groupby("user")["rating"].agg(
        rating_entropy      = rating_entropy,
        extreme_rating_prop = extreme_prop,
        high_rating_prop    = high_prop,
        low_rating_prop     = low_prop,
    )

    # ── Group 4: Interaction diversity ────────────────────────────────────────
    diversity = df.groupby("user").agg(num_unique_items=("item", "nunique"))
    diversity["item_diversity_ratio"] = (
        diversity["num_unique_items"] / basic["num_ratings"]
    )

    # ── Group 5: Deviation from global item means ─────────────────────────────
    # How much does each user deviate from the "expected" rating for each item?
    if global_item_means is None:
        global_item_means = df.groupby("item")["rating"].mean()

    df2 = df.copy()
    df2["item_mean"] = (
        df2["item"].map(global_item_means).fillna(df["rating"].mean())
    )
    df2["deviation"] = df2["rating"] - df2["item_mean"]

    dev_feats = df2.groupby("user")["deviation"].agg(
        mean_deviation     = "mean",
        std_deviation      = "std",
        abs_mean_deviation = lambda x: x.abs().mean(),
    ).fillna(0)

    # ── Group 6: Item popularity features ────────────────────────────────────
    # Do anomalous users target popular or obscure items?
    item_popularity = df.groupby("item")["rating"].count()
    df3 = df.copy()
    df3["item_pop"] = df3["item"].map(item_popularity).fillna(1)

    pop_feats = df3.groupby("user").agg(
        mean_item_popularity = ("item_pop", "mean"),
        std_item_popularity  = ("item_pop", "std"),
    ).fillna(0)

    # ── Group 7: Sequential / burstiness features ─────────────────────────────
    bursty = df.groupby("user")["rating"].agg(
        repeated_rating_prop = repeated_rating_prop,
        consec_diff_std      = consec_diff_std,
    )

    # ── Group 8: SVD latent embeddings ────────────────────────────────────────
    # Build a sparse user–item rating matrix and apply truncated SVD.
    # This captures latent taste patterns that anomalous users may violate.
    all_users = sorted(df["user"].unique())
    user_idx  = {u: i for i, u in enumerate(all_users)}

    rows = df["user"].map(user_idx).values
    cols = np.clip(df["item"].values.astype(int), 0, n_items - 1)
    vals = df["rating"].values.astype(float)

    mat = csr_matrix((vals, (rows, cols)), shape=(len(all_users), n_items))

    if fit_svd:
        svd_model = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=RANDOM_STATE)
        svd_feats = svd_model.fit_transform(mat)
    else:
        svd_feats = svd_model.transform(mat)

    svd_df = pd.DataFrame(
        svd_feats,
        index   = all_users,
        columns = [f"svd_{i}" for i in range(SVD_COMPONENTS)],
    )

    # ── Combine all feature groups ────────────────────────────────────────────
    features = (
        basic
        .join(moments)
        .join(dist)
        .join(diversity)
        .join(dev_feats)
        .join(pop_feats)
        .join(bursty)
        .join(svd_df)
    ).fillna(0)

    return features, global_item_means, svd_model


# Build training features
print("Building training features...")
X_feat, global_item_means, svd_model = build_features(
    ratings, fit_svd=True
)
y_series = labels.loc[X_feat.index]["label"]

print(f"Feature matrix shape : {X_feat.shape}")
print(f"\nFeature groups:")
print(f"  Basic stats (7)      : mean, std, count, min, max, median, range")
print(f"  Higher moments (2)   : skewness, kurtosis")
print(f"  Distribution (4)     : entropy, extreme/high/low props")
print(f"  Diversity (2)        : unique items, diversity ratio")
print(f"  Item deviation (3)   : mean, std, abs deviation from item means")
print(f"  Popularity (2)       : mean/std of item popularity")
print(f"  Sequential (2)       : repeated rating prop, consec diff std")
print(f"  SVD latent (15)      : truncated SVD embeddings")
print(f"  TOTAL: {X_feat.shape[1]} features")

Building training features...
Feature matrix shape : (1100, 37)

Feature groups:
  Basic stats (7)      : mean, std, count, min, max, median, range
  Higher moments (2)   : skewness, kurtosis
  Distribution (4)     : entropy, extreme/high/low props
  Diversity (2)        : unique items, diversity ratio
  Item deviation (3)   : mean, std, abs deviation from item means
  Popularity (2)       : mean/std of item popularity
  Sequential (2)       : repeated rating prop, consec diff std
  SVD latent (15)      : truncated SVD embeddings
  TOTAL: 37 features


## Section 4 — Ensemble Models + Cross-Validation

We train 3 supervised tree-based classifiers using **Stratified 5-Fold CV**:

- **RandomForestClassifier** — bagging + feature subsampling, robust to noise
- **ExtraTreesClassifier** — additional randomization, faster and slightly less correlated with RF
- **GradientBoostingClassifier** — sequential boosting, captures complex interactions

**Class imbalance mitigation:**
- `class_weight='balanced'` in tree models (automatically scales loss)
- **Random oversampling** of minority class per fold (before fitting each model)

**OOF (Out-of-Fold) predictions** are collected for each model and later blended.

In [ ]:
X_arr = X_feat.values
y_arr = y_series.values

# ── Model definitions ─────────────────────────────────────────────────────────
# We instantiate fresh models inside the loop, but define configs here.
MODEL_CONFIGS = {
    "RandomForest": dict(
        estimator = RandomForestClassifier,
        params = dict(
            n_estimators    = 400,
            class_weight    = "balanced",
            max_depth       = 10,
            min_samples_leaf = 3,
            max_features    = "sqrt",
            random_state    = RANDOM_STATE,
            n_jobs          = -1,
        ),
    ),
    "ExtraTrees": dict(
        estimator = ExtraTreesClassifier,
        params = dict(
            n_estimators    = 400,
            class_weight    = "balanced",
            max_depth       = 10,
            min_samples_leaf = 3,
            max_features    = "sqrt",
            random_state    = RANDOM_STATE,
            n_jobs          = -1,
        ),
    ),
    "GradientBoosting": dict(
        estimator = GradientBoostingClassifier,
        params = dict(
            n_estimators  = 200,
            max_depth     = 4,
            learning_rate = 0.05,
            subsample     = 0.8,
            random_state  = RANDOM_STATE,
        ),
    ),
}

model_names = list(MODEL_CONFIGS.keys())
oof_preds   = {name: np.zeros(len(y_arr)) for name in model_names}

# ── Cross-Validation Loop ─────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
fold_scalers = []  # store scalers fitted per fold (for test-time averaging)
fold_models  = {name: [] for name in model_names}  # store trained models

print(f"Starting {N_FOLDS}-Fold Stratified CV...\n")

for fold_idx, (train_idx, val_idx) in enumerate(cv.split(X_arr, y_arr)):
    X_tr, X_va = X_arr[train_idx], X_arr[val_idx]
    y_tr, y_va = y_arr[train_idx], y_arr[val_idx]

    # ── Oversample minority class on training fold ────────────────────────────
    # Duplicate anomalous users to match the number of normal users.
    # We only oversample in the training portion, NEVER in validation.
    pos_idx = np.where(y_tr == 1)[0]
    neg_idx = np.where(y_tr == 0)[0]
    n_to_add = len(neg_idx) - len(pos_idx)
    if n_to_add > 0:
        extra_idx = resample(pos_idx, n_samples=n_to_add, replace=True,
                             random_state=RANDOM_STATE + fold_idx)
        X_tr_aug = np.vstack([X_tr, X_tr[extra_idx]])
        y_tr_aug = np.concatenate([y_tr, y_tr[extra_idx]])
    else:
        X_tr_aug, y_tr_aug = X_tr, y_tr

    # ── Scale features ─────────────────────────────────────────────────────────
    scaler = StandardScaler().fit(X_tr_aug)
    X_tr_s = scaler.transform(X_tr_aug)
    X_va_s = scaler.transform(X_va)
    fold_scalers.append(scaler)

    fold_aucs = {}
    for name, cfg in MODEL_CONFIGS.items():
        clf = cfg["estimator"](**cfg["params"])
        clf.fit(X_tr_s, y_tr_aug)
        oof_preds[name][val_idx] = clf.predict_proba(X_va_s)[:, 1]
        fold_aucs[name] = roc_auc_score(y_va, oof_preds[name][val_idx])
        fold_models[name].append(clf)   # keep for final test inference

    avg_fold_auc = np.mean(list(fold_aucs.values()))
    print(f"Fold {fold_idx+1}/{N_FOLDS}  "
          + "  ".join(f"{n}: {a:.4f}" for n, a in fold_aucs.items())
          + f"  │  Avg: {avg_fold_auc:.4f}")

print("\n" + "="*60)
print("OVERALL OOF AUC (full dataset):")
for name, oof in oof_preds.items():
    print(f"  {name:<20} AUC = {roc_auc_score(y_arr, oof):.4f}")

Starting 5-Fold Stratified CV...

Fold 1/5  RandomForest: 0.9417  ExtraTrees: 0.8902  GradientBoosting: 0.9518  │  Avg: 0.9279
Fold 2/5  RandomForest: 0.9600  ExtraTrees: 0.9605  GradientBoosting: 0.9520  │  Avg: 0.9575
Fold 3/5  RandomForest: 0.9620  ExtraTrees: 0.9583  GradientBoosting: 0.9625  │  Avg: 0.9609
Fold 4/5  RandomForest: 0.9260  ExtraTrees: 0.9275  GradientBoosting: 0.9210  │  Avg: 0.9248
Fold 5/5  RandomForest: 0.9055  ExtraTrees: 0.9080  GradientBoosting: 0.9195  │  Avg: 0.9110

OVERALL OOF AUC (full dataset):
  RandomForest         AUC = 0.9388
  ExtraTrees           AUC = 0.9289
  GradientBoosting     AUC = 0.9424


## Section 5 — Ensemble Weight Optimization & Blending

We find the optimal non-negative weights for blending the OOF predictions by maximizing AUC on the OOF set using Nelder-Mead optimization.

In [ ]:
oof_list = [oof_preds[name] for name in model_names]
n_models = len(oof_list)

def neg_auc_blend(weights, oof_list, y):
    """Objective: minimize -AUC of the weighted blend."""
    w = np.maximum(weights, 0)       # force non-negative
    w = w / (w.sum() + 1e-9)         # normalize to sum = 1
    blend = sum(wi * oi for wi, oi in zip(w, oof_list))
    return -roc_auc_score(y, blend)

# Start from equal weights
w0 = np.ones(n_models) / n_models

result = minimize(
    neg_auc_blend,
    w0,
    args=(oof_list, y_arr),
    method="Nelder-Mead",
    options={"maxiter": 10_000, "xatol": 1e-7, "fatol": 1e-7},
)

opt_weights = np.maximum(result.x, 0)
opt_weights = opt_weights / opt_weights.sum()

print("Optimized ensemble weights:")
for name, w in zip(model_names, opt_weights):
    print(f"  {name:<22} weight = {w:.4f}")

# Blended OOF prediction
blended_oof = sum(w * o for w, o in zip(opt_weights, oof_list))
blended_auc = roc_auc_score(y_arr, blended_oof)

print(f"\nBlended OOF AUC  : {blended_auc:.4f}")
print(f"Baseline AUC     : 0.8573")
print(f"Improvement      : +{blended_auc - 0.8573:.4f}")

Optimized ensemble weights:
  RandomForest           weight = 0.2016
  ExtraTrees             weight = 0.1986
  GradientBoosting       weight = 0.5998

Blended OOF AUC  : 0.9448
Baseline AUC     : 0.8573
Improvement      : +0.0875


## Section 6 — Full Evaluation Report

In [ ]:
# Find optimal classification threshold (maximise F1 on OOF)
thresholds = np.linspace(0.1, 0.9, 100)
f1_scores  = [f1_score(y_arr, (blended_oof >= t).astype(int)) for t in thresholds]
best_thresh = thresholds[np.argmax(f1_scores)]

y_pred = (blended_oof >= best_thresh).astype(int)

print("=" * 50)
print("FULL EVALUATION REPORT (OOF predictions)")
print("=" * 50)
print(f"\nOptimal threshold  : {best_thresh:.3f}")
print(f"AUC                : {roc_auc_score(y_arr, blended_oof):.4f}")
print(f"Precision          : {precision_score(y_arr, y_pred):.4f}")
print(f"Recall             : {recall_score(y_arr, y_pred):.4f}")
print(f"F1 Score           : {f1_score(y_arr, y_pred):.4f}")

print("\n--- Baseline Comparison ---")
print(f"{'Metric':<15} {'Baseline':>10} {'Improved':>10} {'Delta':>10}")
print("-" * 50)
for metric, baseline, improved in [
    ("AUC",       0.8573, roc_auc_score(y_arr, blended_oof)),
    ("Precision", 0.75,   precision_score(y_arr, y_pred)),
    ("Recall",    0.45,   recall_score(y_arr, y_pred)),
    ("F1",        0.5625, f1_score(y_arr, y_pred)),
]:
    delta = improved - baseline
    print(f"{metric:<15} {baseline:>10.4f} {improved:>10.4f} {delta:>+10.4f}")

print("\n--- Per-Model OOF AUC ---")
for name, oof in oof_preds.items():
    print(f"  {name:<22} {roc_auc_score(y_arr, oof):.4f}")

FULL EVALUATION REPORT (OOF predictions)

Optimal threshold  : 0.553
AUC                : 0.9448
Precision          : 0.8750
Recall             : 0.5600
F1 Score           : 0.6829

--- Baseline Comparison ---
Metric            Baseline   Improved      Delta
--------------------------------------------------
AUC                 0.8573     0.9448    +0.0875
Precision           0.7500     0.8750    +0.1250
Recall              0.4500     0.5600    +0.1100
F1                  0.5625     0.6829    +0.1204

--- Per-Model OOF AUC ---
  RandomForest           0.9388
  ExtraTrees             0.9289
  GradientBoosting       0.9424


## Section 7 — Final Prediction on Test Set

We use **fold-averaged predictions** for robustness:
- For each of the 5 folds we have a trained model + scaler
- We apply each to the test set and average the probabilities
- Final ensemble score = weighted average across all models and folds

In [ ]:
# ── Load and process test data ────────────────────────────────────────────────
data_test   = np.load("first_batch.npz")
X_test_raw  = data_test["X"]
df_test     = pd.DataFrame(X_test_raw, columns=["user", "item", "rating"])

print(f"Test interactions  : {len(df_test):,}")
print(f"Test unique users  : {df_test['user'].nunique()}")

# Build features using the SAME global_item_means and svd_model from training
print("Building test features...")
X_test_feat, _, _ = build_features(
    df_test,
    global_item_means = global_item_means,   # fitted on training data
    svd_model         = svd_model,           # fitted on training data
    fit_svd           = False,               # MUST be False for test
)
X_test_arr = X_test_feat.values
test_users  = X_test_feat.index.values

print(f"Test feature matrix: {X_test_arr.shape}")

# ── Fold-averaged predictions per model ───────────────────────────────────────
# For each model: average the predictions from all 5 fold-trained versions.
# Each fold used its own scaler → we apply the matching scaler.
test_preds_per_model = {}

for name in model_names:
    fold_test_scores = []
    for fold_idx, (clf, scaler) in enumerate(zip(fold_models[name], fold_scalers)):
        X_test_s = scaler.transform(X_test_arr)
        scores   = clf.predict_proba(X_test_s)[:, 1]
        fold_test_scores.append(scores)
    # Average over 5 folds
    test_preds_per_model[name] = np.mean(fold_test_scores, axis=0)

# ── Weighted ensemble blend ───────────────────────────────────────────────────
test_oof_list    = [test_preds_per_model[name] for name in model_names]
final_scores     = sum(w * o for w, o in zip(opt_weights, test_oof_list))

print(f"\nGenerated {len(final_scores)} anomaly scores for test users.")
print(f"Score range: [{final_scores.min():.4f}, {final_scores.max():.4f}]")
print(f"Mean score : {final_scores.mean():.4f}")

# Preview top anomalous users
results_df = pd.DataFrame({
    "user"              : test_users,
    "anomaly_score"     : final_scores,
}).sort_values("anomaly_score", ascending=False)

print("\nTop 10 most anomalous users:")
print(results_df.head(10).to_string(index=False))

Test interactions  : 167,493
Test unique users  : 1100
Building test features...
Test feature matrix: (1100, 37)

Generated 1100 anomaly scores for test users.
Score range: [0.0059, 0.9924]
Mean score : 0.1371

Top 10 most anomalous users:
 user  anomaly_score
 2726       0.992437
 2948       0.992291
 3489       0.992110
 2861       0.991881
 2780       0.991694
 2991       0.991054
 2747       0.990991
 2758       0.990460
 3362       0.990177
 3102       0.989676


## Section 8 — Save Submission

In [ ]:
# ── Save predictions in required format ───────────────────────────────────────
# The key MUST be named 'predictions' (as per task specification)
np.savez("submission_batch1.npz", predictions=final_scores)

# Verify the saved file
check = np.load("submission_batch1.npz")
assert "predictions" in check, "ERROR: 'predictions' key missing!"
assert len(check["predictions"]) == len(test_users), "ERROR: Length mismatch!"

print(f"Submission saved to: submission_batch1.npz")
print(f"Key: 'predictions'")
print(f"Number of predictions: {len(check['predictions'])}")
print(f"Score range: [{check['predictions'].min():.4f}, {check['predictions'].max():.4f}]")
print("\n✓ Submission verified successfully.")

Submission saved to: submission_batch1.npz
Key: 'predictions'
Number of predictions: 1100
Score range: [0.0059, 0.9924]

✓ Submission verified successfully.


## Summary of Improvements

| Component | Baseline | Improved |
|---|---|---|
| Features | 5 (basic stats) | 33 (stats + moments + entropy + SVD + ...) |
| Models | RandomForest only | RF + ExtraTrees + GradientBoosting |
| Imbalance handling | `class_weight='balanced'` | `class_weight` + random oversampling |
| Cross-validation | None | Stratified 5-Fold |
| Ensemble blending | None | Nelder-Mead weight optimization on OOF |
| OOF AUC | 0.8573 | **~0.944** |

### Key Feature Insights
- **Rating entropy**: anomalous users often have very low entropy (rating everything the same) or very high entropy (random ratings)
- **Item deviation**: anomalous users systematically deviate from community consensus
- **SVD embeddings**: capture latent taste signatures; anomalous users sit in unusual regions of latent space
- **Repeated rating proportion**: bot-like behavior produces many consecutive identical ratings
- **Extreme rating proportion**: shills often rate only 5 stars (or only 0 stars)